Import everything that is needed.

In [1]:
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np

Import the data of the iris dataset from sklearn.

In [2]:
iris = load_iris()
features = iris['data']
labels = iris['target']
class_names = iris['target_names']
feature_names = iris['feature_names']

print(f"Features: {feature_names}, Classes: {class_names}")

Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)'], Classes: ['setosa' 'versicolor' 'virginica']


Some data processing.

In [3]:
# One-hot encoding
n_values = np.max(labels) + 1
labels_onehot = np.eye(n_values)[labels]

# Split the data 
features_train, features_test, labels_train, labels_test = train_test_split(features, labels_onehot, test_size=0.3, random_state=19)

# Normalize the training data
scaler = MinMaxScaler()
scaler.fit(features_train)
features_train = scaler.transform(features_train)

# Apply the above normalization to the testing data
features_test = scaler.transform(features_test)

dataset = torch.utils.data.TensorDataset(torch.tensor(features_train, dtype=torch.float32),torch.tensor(labels_train,dtype=torch.float32))
dataloader = torch.utils.data.DataLoader(dataset,batch_size=10)

features_test = torch.tensor(features_test, dtype=torch.float32)
labels_test = torch.tensor(labels_test, dtype=torch.float32)

Define a generic training function that can be used for different networks.

In [ ]:
# Generic training function
def train(net, dataloader, features_test, labels_test, epochs=10, lr=0.05):
    #optim = torch.optim.SGD(net.parameters(),lr=lr)
    optim = torch.optim.Adam(net.parameters(),lr=lr) # Change the optimizer made a big difference
    for ep in range(epochs):
        for (x,y) in dataloader:
            z = net(x)
            loss = torch.nn.CrossEntropyLoss() 
            output = loss(z,y)
            optim.zero_grad()
            output.backward()
            optim.step()
        acc = (torch.argmax(net(features_test), dim=1)==torch.argmax(labels_test, dim = 1)).float().mean() 
        print(f"Epoch {ep}: last batch loss = {loss}, val acc = {acc}")

Define several neural networks and train them to compare results.

In [10]:
# Train a one-layer network
net1 = torch.nn.Linear(4,3)
train(net1, dataloader, features_test, labels_test)

# The accuracy on the testing data is not high.

Epoch 0: last batch loss = CrossEntropyLoss(), val acc = 0.3333333432674408
Epoch 1: last batch loss = CrossEntropyLoss(), val acc = 0.6888889074325562
Epoch 2: last batch loss = CrossEntropyLoss(), val acc = 0.6666666865348816
Epoch 3: last batch loss = CrossEntropyLoss(), val acc = 0.6666666865348816
Epoch 4: last batch loss = CrossEntropyLoss(), val acc = 0.8222222328186035
Epoch 5: last batch loss = CrossEntropyLoss(), val acc = 0.9111111164093018
Epoch 6: last batch loss = CrossEntropyLoss(), val acc = 0.9111111164093018
Epoch 7: last batch loss = CrossEntropyLoss(), val acc = 0.9111111164093018
Epoch 8: last batch loss = CrossEntropyLoss(), val acc = 0.9333333373069763
Epoch 9: last batch loss = CrossEntropyLoss(), val acc = 0.9333333373069763


In [12]:
# Train a two-layer network
net2 = torch.nn.Sequential(torch.nn.Linear(4,30),torch.nn.ReLU(),torch.nn.Linear(30,3)) 
train(net2, dataloader, features_test, labels_test)

# I tried different sizes for the hidden layer and it seems that ~30 is the sweet spot. This way the accuracy on the testing data is always above 0.95.

Epoch 0: last batch loss = CrossEntropyLoss(), val acc = 0.6666666865348816
Epoch 1: last batch loss = CrossEntropyLoss(), val acc = 0.8888888955116272
Epoch 2: last batch loss = CrossEntropyLoss(), val acc = 0.8222222328186035
Epoch 3: last batch loss = CrossEntropyLoss(), val acc = 0.8444444537162781
Epoch 4: last batch loss = CrossEntropyLoss(), val acc = 0.8888888955116272
Epoch 5: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509
Epoch 6: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509
Epoch 7: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509
Epoch 8: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509
Epoch 9: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509


In [24]:
# Train a three-layer network
net3 = torch.nn.Sequential(torch.nn.Linear(4,20),torch.nn.ReLU(),torch.nn.Linear(20,10), torch.nn.ReLU(),torch.nn.Linear(10,3)) 
train(net3, dataloader, features_test, labels_test)

# The accuracy on the testing data is less stable than that of a two-layer network no matter what the sizes of the hidden layers are. It seems that two hidden layers are unnecessary.

Epoch 0: last batch loss = CrossEntropyLoss(), val acc = 0.644444465637207
Epoch 1: last batch loss = CrossEntropyLoss(), val acc = 0.9777777791023254
Epoch 2: last batch loss = CrossEntropyLoss(), val acc = 0.6666666865348816
Epoch 3: last batch loss = CrossEntropyLoss(), val acc = 0.7111111283302307
Epoch 4: last batch loss = CrossEntropyLoss(), val acc = 0.9777777791023254
Epoch 5: last batch loss = CrossEntropyLoss(), val acc = 0.9777777791023254
Epoch 6: last batch loss = CrossEntropyLoss(), val acc = 0.9777777791023254
Epoch 7: last batch loss = CrossEntropyLoss(), val acc = 0.9555555582046509
Epoch 8: last batch loss = CrossEntropyLoss(), val acc = 0.9111111164093018
Epoch 9: last batch loss = CrossEntropyLoss(), val acc = 0.8888888955116272
